# 🚀 MyUPI Voice + Action Assistant on Google Colab (GPU)

This notebook runs the complete **MyUPI Voice Assistant** on a free Google Colab GPU with full **Voice-In (ASR)** and **Voice-Out (TTS)** enabled!

### Step 1: Check GPU
Ensure you are connected to a GPU runtime (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!nvidia-smi

### Step 2: Install System and Python Dependencies
(Works universally on Python 3.10, 3.11, 3.12, and 3.13)

In [ ]:
!apt-get install -y ffmpeg
!pip install -r requirements.txt
!pip install faster-whisper edge-tts pycloudflared

### Step 3: Pre-cache Hugging Face ASR Model on GPU
Downloads Whisper (FP16 on CUDA) directly from Hugging Face.

In [ ]:
from faster_whisper import WhisperModel
print("📥 Downloading Faster-Whisper ASR model from Hugging Face...")
asr_model = WhisperModel('small', device='cuda', compute_type='float16', download_root='models/whisper')
print("✅ Hugging Face ASR model cached and ready on CUDA GPU!")

### Step 4: Configure `config/config.yaml` for Local GPU Voice Pipeline

In [ ]:
import yaml

with open('config/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# Set ASR to faster_whisper on GPU
cfg['asr']['provider'] = 'faster_whisper'
cfg['asr']['model_authoritative'] = 'small'
cfg['asr']['device'] = 'cuda'
cfg['asr']['compute_type'] = 'float16'

# Set TTS to Neural Indian English / Hindi
cfg['tts']['provider'] = 'edge_tts'
cfg['tts']['voice'] = 'en-IN-NeerjaNeural'

with open('config/config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("✅ config.yaml updated for GPU Voice Pipeline!")

### Step 5: Launch Server with Public HTTPS URL (Cloudflare Tunnel)
Click the generated `trycloudflare.com` public URL to open the MyUPI Web App in your browser and test voice!

In [ ]:
from pycloudflared import try_cloudflare

# Start Cloudflare tunnel on port 8000
tunnel_url = try_cloudflare(port=8000)
print(f"\n🎉 YOUR PUBLIC APP URL: {tunnel_url.tunnel}\n")
print("Open this link on your phone or laptop to speak directly with the AI!\n")

# Launch FastAPI server
!uvicorn app.main:app --host 0.0.0.0 --port 8000